# BlEmoRe — CMAFNet (Cross-Modal Attentive Fusion Network)

**Key improvements over the previous MLP-concat approach:**
1. 🔀 **Cross-modal transformer attention** — modalities interact via self-attention, not just concatenation
2. 🔍 **Auto-discovers ALL feature modalities** — OpenFace, CLIP, ImageBind, HiCMAE, DINOv2 + prior 4
3. 📉 **Focal CE loss + label smoothing** — better for imbalanced 21-class problem
4. 🎲 **Feature-space Mixup augmentation** — virtual samples for the small dataset
5. 🎯 **Attention pooling** — context-aware modality aggregation instead of simple mean
6. 🏆 **Larger ensemble** — 5 folds × 4 architecture variants = up to 20 models


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Copy data from Drive to fast local SSD — adjust paths if yours differ
!cp -rn "/content/drive/MyDrive/blemore_project-backup/data/pre_extracted_train/pre_extracted_train_data/" "/content/pre_extracted_train_data"
!cp -rn "/content/drive/MyDrive/blemore_project-backup/data/pre_extracted_test/pre_extracted_test_data/" "/content/pre_extracted_test_data"


In [ ]:
import os, math, random, json, zipfile, copy, time, warnings, gc
from contextlib import nullcontext
from ast import literal_eval
from typing import List, Dict, Optional, Tuple
from itertools import combinations

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTHONHASHSEED"] = "42"
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from sklearn.model_selection import GroupKFold

# ── PATHS (adjust if your Drive layout differs) ─────────────────────────
ROOT_TRAIN = "/content/pre_extracted_train_data"
ROOT_TEST  = "/content/pre_extracted_test_data"
OUTPUT_DIR = "/content/drive/MyDrive/blemore_project-backup/blemore_output-cmaf-01"
META_CSV   = "/content/drive/MyDrive/blemore_project-backup/blemore_output-17/train_metadata.csv"

for d in [OUTPUT_DIR, f"{OUTPUT_DIR}/models", f"{OUTPUT_DIR}/epoch_ckpts"]:
    os.makedirs(d, exist_ok=True)

# ── REPRODUCIBILITY ─────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name}  VRAM: {p.total_memory/1e9:.1f} GB")

# ── TRAINING HYPER-PARAMETERS ────────────────────────────────────────────
FOLDS           = 5
FOLDS_TO_TRAIN  = [0, 1, 2, 3, 4]
EPOCHS          = 80
SWA_START       = 60      # epoch to begin Stochastic Weight Averaging
SWA_LR          = 2e-5
WARMUP_EPOCHS   = 5
BATCH_SIZE      = 128
NUM_WORKERS     = 0
WEIGHT_DECAY    = 5e-3
GRAD_CLIP       = 1.0
MIXUP_ALPHA     = 0.3     # feature-space mixup Beta(alpha, alpha) strength
MIX_SAMPLE_MULT = 4.0    # oversampling multiplier for blended samples
ENSEMBLE_TOPK   = 12     # top-K checkpoints used in test ensemble

# ── ARCHITECTURE VARIANTS ────────────────────────────────────────────────
# mod_proj : dim each modality is projected to (= transformer token size)
# n_heads  : number of attention heads in cross-modal transformer
# n_layers : number of cross-modal transformer blocks
MODEL_VARIANTS = [
    dict(name="cmaf_s",  mod_proj=64,  n_heads=4, n_layers=2, dropout=0.45, lr=4e-4),
    dict(name="cmaf_m",  mod_proj=96,  n_heads=4, n_layers=2, dropout=0.40, lr=3e-4),
    dict(name="cmaf_l",  mod_proj=64,  n_heads=4, n_layers=3, dropout=0.45, lr=3e-4),
    dict(name="cmaf_xl", mod_proj=96,  n_heads=4, n_layers=3, dropout=0.50, lr=3e-4),
]
print(f"Variants: {[v['name'] for v in MODEL_VARIANTS]}")
print(f"Total models to train: {len(MODEL_VARIANTS) * len(FOLDS_TO_TRAIN)}")


Device: cuda
GPU: Tesla T4  VRAM: 15.6 GB
Variants: ['cmaf_s', 'cmaf_m', 'cmaf_l', 'cmaf_xl']
Total models to train: 20


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 2 — Emotion taxonomy & 21-class label space           ║
# ╚══════════════════════════════════════════════════════════════╝
EMO_CODES  = ['ang', 'disg', 'fea', 'hap', 'sad', 'neu']
EMO2IDX    = {e: i for i, e in enumerate(EMO_CODES)}
N_EMO      = len(EMO_CODES)

SINGLE_COMBOS  = [(e,) for e in EMO_CODES]                              # 6 singles
BLENDED_COMBOS = [tuple(sorted(p)) for p in combinations(EMO_CODES, 2)] # 15 blends
ALL_COMBOS     = SINGLE_COMBOS + BLENDED_COMBOS                         # 21 total
N_COMBO        = len(ALL_COMBOS)
COMBO2IDX      = {c: i for i, c in enumerate(ALL_COMBOS)}
N_SAL          = 3   # salience classes: {0=first_dom 70/30, 1=equal 50/50, 2=second_dom 30/70}

print(f"N_COMBO={N_COMBO}")
print("Singles: ", ALL_COMBOS[:6])
print("Blended (first 5):", ALL_COMBOS[6:11], "...")


N_COMBO=21
Singles:  [('ang',), ('disg',), ('fea',), ('hap',), ('sad',), ('neu',)]
Blended (first 5): [('ang', 'disg'), ('ang', 'fea'), ('ang', 'hap'), ('ang', 'sad'), ('ang', 'neu')] ...


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 3 — Feature utilities                                 ║
# ╚══════════════════════════════════════════════════════════════╝
def _pool(arr) -> Optional[np.ndarray]:
    """Temporal mean+max pooling over frame axis."""
    try: arr = np.asarray(arr, dtype=np.float32)
    except: return None
    arr = np.nan_to_num(arr, nan=0., posinf=0., neginf=0.)
    if arr.ndim == 0 or arr.size == 0: return None
    if arr.ndim == 1: return np.concatenate([arr, arr])  # single frame: replicate
    x = arr.reshape(-1, arr.shape[-1])
    return np.concatenate([x.mean(0), x.max(0)])

def _fix_dim(v: np.ndarray, d: int) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).reshape(-1)
    if v.shape[0] == d: return v
    if v.shape[0]  > d: return v[:d]
    return np.concatenate([v, np.zeros(d - v.shape[0], dtype=np.float32)])

def _find_folder(root: str, mod: str) -> Optional[str]:
    """Try common name variants to locate a modality folder."""
    candidates = [
        mod,
        mod.replace('_train', '_test'),
        mod.replace('_test',  '_train'),
        mod.replace('_train', ''),
        mod.replace('_test',  ''),
        mod + '_train',
    ]
    for c in candidates:
        if c and os.path.isdir(os.path.join(root, c)):
            return c
    return None

def _raw_pool(root: str, mod_dir: str, fname: str) -> Optional[np.ndarray]:
    base = os.path.splitext(os.path.basename(fname))[0]
    for path in [f"{root}/{mod_dir}/{base}.npy",
                 f"{root}/{mod_dir}/{base}.mov.npy",
                 f"{root}/{mod_dir}/{fname}.npy"]:
        if not os.path.exists(path): continue
        try:
            p = _pool(np.load(path, allow_pickle=True))
            if p is not None: return p
        except Exception as e:
            warnings.warn(f"[_raw_pool] {path}: {e}")
    return None

def load_feature(root, mod_dir, fname, dim):
    p = _raw_pool(root, mod_dir, fname)
    return _fix_dim(p, dim) if p is not None else None

def _parse_vec(x, n):
    out = np.zeros(n, dtype=np.float32)
    if isinstance(x, (list, tuple, np.ndarray)):
        a = np.asarray(x, dtype=np.float32).reshape(-1)
        out[:min(n, len(a))] = a[:min(n, len(a))]
    elif isinstance(x, str):
        try: return _parse_vec(literal_eval(x), n)
        except: pass
    return out


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 4 — Modality discovery, normalization stats, cache    ║
# ║  NEW: scans for ALL feature types (not just 4)              ║
# ╚══════════════════════════════════════════════════════════════╝

# All possible feature modalities from the Zenodo dataset
ALL_POSSIBLE_MODS = [
    # Temporal video features
    'VideoMAEv2_train', 'VideoSwinTransformer_train',
    # Facial action units — very informative for emotion
    'OpenFace2_train', 'OpenFace3_train',
    # Audio representations
    'WavLM_large_train', 'HuBERT_large_train',
    # Vision-language / multi-modal
    'ImageBind_train', 'CLIP_train',
    # Self-supervised vision
    'DINOv2_train', 'DINOv3_train',
    # Multimodal emotion-specific
    'HiCMAE_train',
]

print("Scanning TRAIN root for available modalities:")
selected_mods, seen_dirs = [], set()
for m in ALL_POSSIBLE_MODS:
    folder = _find_folder(ROOT_TRAIN, m)
    if folder and folder not in seen_dirs:
        seen_dirs.add(folder)
        selected_mods.append(m)
        print(f"  ✓ {m:30s} -> {folder}")

assert selected_mods, "No modalities found! Check ROOT_TRAIN."
print(f"\n→ {len(selected_mods)} modalities found: {selected_mods}")

# Training sample filenames (from first available modality folder)
first_folder  = _find_folder(ROOT_TRAIN, selected_mods[0])
train_samples = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(f"{ROOT_TRAIN}/{first_folder}")
    if f.lower().endswith('.npy')
)
print(f"Training samples: {len(train_samples)}")

# ── Normalization statistics ──────────────────────────────────────────
def compute_stats(root, modalities, samples, n=500):
    stats = {}
    for mod in modalities:
        mod_dir = _find_folder(root, mod)
        if not mod_dir: continue
        probe_dim = next(
            (p.shape[0] for s in samples[:n]
             if (p := _raw_pool(root, mod_dir, s)) is not None), None)
        if probe_dim is None: continue
        feats = [f for s in samples[:n]
                 if (f := load_feature(root, mod_dir, s, probe_dim)) is not None]
        if not feats: continue
        mat = np.stack(feats).astype(np.float32)
        std = mat.std(0); std[std < 1e-6] = 1.
        stats[mod] = {"dim": probe_dim, "mean": mat.mean(0), "std": std}
        print(f"  {mod}: dim={probe_dim}  n={len(feats)}")
    print(f"Total feat dim: {sum(v['dim'] for v in stats.values())}")
    return stats

cache_path = f"{OUTPUT_DIR}/modality_stats.npy"
if os.path.exists(cache_path):
    stats    = np.load(cache_path, allow_pickle=True).item()
    new_mods = [m for m in selected_mods if m not in stats]
    if new_mods:
        print(f"Computing stats for new mods: {new_mods}")
        stats.update(compute_stats(ROOT_TRAIN, new_mods, train_samples))
        np.save(cache_path, stats, allow_pickle=True)
    print("Loaded stats:", {m: v['dim'] for m, v in stats.items()})
else:
    stats = compute_stats(ROOT_TRAIN, selected_mods, train_samples)
    np.save(cache_path, stats, allow_pickle=True)

selected_mods = [m for m in selected_mods if m in stats]
print(f"\nFinal modalities ({len(selected_mods)}): {selected_mods}")
total_feat_dim = sum(stats[m]['dim'] for m in selected_mods)
print(f"Total feature dimension: {total_feat_dim}")

# ── RAM cache (z-scored features) ────────────────────────────────────
def build_ram_cache(root_dir, modalities, stats, filenames, desc="Cache"):
    mod_dirs = {m: _find_folder(root_dir, m) or m for m in modalities}
    dims  = {m: int(stats[m]['dim']) for m in modalities}
    means = {m: np.asarray(stats[m]['mean'], dtype=np.float32) for m in modalities}
    stds  = {m: np.asarray(stats[m]['std'],  dtype=np.float32) for m in modalities}
    cache = {}
    for fname in tqdm(filenames, desc=desc):
        for m in modalities:
            feat = load_feature(root_dir, mod_dirs[m], fname, dims[m])
            if feat is not None:
                cache[(m, fname)] = (feat - means[m]) / (stds[m] + 1e-6)
    n_total = len(filenames) * len(modalities)
    hit_pct = 100 * len(cache) / n_total if n_total else 0
    ram_mb  = sum(v.nbytes for v in cache.values()) / 1e6
    print(f"  {desc}: {len(cache)}/{n_total} ({hit_pct:.1f}%)  RAM: {ram_mb:.1f} MB")
    return cache

print("\nBuilding TRAIN cache ...")
train_cache = build_ram_cache(ROOT_TRAIN, selected_mods, stats, train_samples, "TRAIN")

test_folder  = _find_folder(ROOT_TEST, selected_mods[0])
test_samples = sorted(
    os.path.splitext(f)[0]
    for f in os.listdir(f"{ROOT_TEST}/{test_folder}")
    if f.lower().endswith('.npy')
)
print("Building TEST cache ...")
test_cache = build_ram_cache(ROOT_TEST, selected_mods, stats, test_samples, "TEST")
print(f"Test samples: {len(test_samples)}")


Scanning TRAIN root for available modalities:
  ✓ VideoMAEv2_train               -> VideoMAEv2_train
  ✓ VideoSwinTransformer_train     -> VideoSwinTransformer_train
  ✓ WavLM_large_train              -> WavLM_large_train
  ✓ HuBERT_large_train             -> HuBERT_large_train

→ 4 modalities found: ['VideoMAEv2_train', 'VideoSwinTransformer_train', 'WavLM_large_train', 'HuBERT_large_train']
Training samples: 2456
  VideoMAEv2_train: dim=2816  n=500
  VideoSwinTransformer_train: dim=1536  n=500
  WavLM_large_train: dim=2048  n=500
  HuBERT_large_train: dim=2048  n=500
Total feat dim: 8448

Final modalities (4): ['VideoMAEv2_train', 'VideoSwinTransformer_train', 'WavLM_large_train', 'HuBERT_large_train']
Total feature dimension: 8448

Building TRAIN cache ...


TRAIN: 100%|██████████| 2456/2456 [00:30<00:00, 80.19it/s]


  TRAIN: 9824/9824 (100.0%)  RAM: 83.0 MB
Building TEST cache ...


TEST: 100%|██████████| 594/594 [00:07<00:00, 81.51it/s] 

  TEST: 2267/2376 (95.4%)  RAM: 19.2 MB
Test samples: 594


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 5 — Metadata with 21-class + 3-class salience labels  ║
# ╚══════════════════════════════════════════════════════════════╝
def _is_miss(x):
    if x is None: return True
    try: return pd.isna(x)
    except: return False

def _emo(x):
    if _is_miss(x): return None
    s = str(x).strip().lower()
    return s if s in EMO2IDX else None

def _fl(x):
    if _is_miss(x): return None
    try: return float(x)
    except: return None

def build_sample_labels(row) -> Dict:
    e1 = _emo(row.get('emotion_1')) or 'neu'
    e2 = _emo(row.get('emotion_2'))
    s1 = _fl(row.get('emotion_1_salience'))
    s2 = _fl(row.get('emotion_2_salience'))
    is_blend = (int(row.get('mix', 0)) == 1) or (e2 in EMO2IDX)

    if not is_blend or e2 not in EMO2IDX:
        combo     = (e1,)
        combo_idx = COMBO2IDX.get(combo, 0)
        sal_idx   = 0
        sal_mask  = 0.
        pres_list = [1. if e == e1 else 0. for e in EMO_CODES]
    else:
        ea, eb    = tuple(sorted([e1, e2]))
        combo     = (ea, eb)
        combo_idx = COMBO2IDX.get(combo, N_COMBO - 1)
        if s1 is None or s2 is None: s1, s2 = 50., 50.
        t = s1 + s2; s1, s2 = s1 / t * 100, s2 / t * 100
        sal_ea    = s1 if e1 == ea else s2
        sal_idx   = 0 if sal_ea >= 65 else (2 if sal_ea <= 35 else 1)
        sal_mask  = 1.
        pres_list = [1. if e in (ea, eb) else 0. for e in EMO_CODES]

    return dict(combo_idx=combo_idx, sal_idx=sal_idx,
                sal_mask=float(is_blend), pres_list=pres_list, is_blend=int(is_blend))

def load_metadata(csv_path):
    df = pd.read_csv(csv_path)
    if 'filename' not in df.columns and 'fname' in df.columns:
        df = df.rename(columns={'fname': 'filename'})
    label_rows = [build_sample_labels(r) for _, r in df.iterrows()]
    for k in ['combo_idx', 'sal_idx', 'sal_mask', 'pres_list', 'is_blend']:
        df[k] = [lr[k] for lr in label_rows]
    if 'fold' not in df.columns or df['fold'].isnull().all():
        gc_col = 'video_id' if 'video_id' in df.columns else 'filename'
        df['fold'] = -1
        for fi, (_, vi) in enumerate(
                GroupKFold(n_splits=FOLDS).split(df, groups=df[gc_col].values)):
            df.loc[vi, 'fold'] = fi
    else:
        df['fold'] = df['fold'].astype(int)
    return df

meta_df = load_metadata(META_CSV)
n_b = int(meta_df['is_blend'].sum())
print(f"Rows: {len(meta_df)}  Blended: {n_b} ({100*n_b/len(meta_df):.1f}%)")
combo_counts = meta_df['combo_idx'].value_counts().sort_index()
print(f"Unique combos used: {len(combo_counts)}/21  Least common: {combo_counts.min()}")


Rows: 2456  Blended: 1326 (54.0%)
Unique combos used: 16/21  Least common: 126


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 6 — Dataset                                           ║
# ╚══════════════════════════════════════════════════════════════╝
class BlEmoDataset(Dataset):
    def __init__(self, meta_df, modalities, fold, split, stats,
                 ram_cache, modality_drop=0.0, feat_noise=0.01):
        assert split in ('train', 'val')
        df = (meta_df[meta_df['fold'] != fold] if split == 'train'
              else meta_df[meta_df['fold'] == fold])
        self.df       = df.reset_index(drop=True)
        self.mods     = modalities
        self.is_train = (split == 'train')
        self.mod_drop = modality_drop
        self.noise    = feat_noise
        self.cache    = ram_cache
        self.dims     = [int(stats[m]['dim']) for m in modalities]
        self.feat_len = sum(self.dims)
        print(f"  [{split} fold={fold}] n={len(self.df)}  feat_len={self.feat_len}")

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row    = self.df.iloc[idx]
        fname  = str(row['filename'])
        drop_m = (random.choice(self.mods)
                  if self.is_train and random.random() < self.mod_drop else None)
        feats  = []
        for m, dim in zip(self.mods, self.dims):
            if m == drop_m:
                vec = np.zeros(dim, dtype=np.float32)
            else:
                vec = self.cache.get((m, fname))
                if vec is None:
                    vec = np.zeros(dim, dtype=np.float32)
                else:
                    vec = vec.copy()
                    if self.is_train and self.noise > 0:
                        vec += np.random.randn(dim).astype(np.float32) * self.noise
            feats.append(vec.astype(np.float32))
        return {
            "features"  : torch.from_numpy(np.concatenate(feats)),
            "combo_idx" : torch.tensor(int(row['combo_idx']),  dtype=torch.long),
            "sal_idx"   : torch.tensor(int(row['sal_idx']),    dtype=torch.long),
            "sal_mask"  : torch.tensor(float(row['sal_mask']), dtype=torch.float32),
            "pres_list" : torch.tensor(_parse_vec(row['pres_list'], N_EMO), dtype=torch.float32),
            "is_blend"  : torch.tensor(int(row['is_blend']),   dtype=torch.long),
            "fname"     : fname,
        }


class BlEmoTestDataset(Dataset):
    def __init__(self, filenames, modalities, stats, ram_cache):
        self.filenames = filenames; self.mods = modalities
        self.dims  = [int(stats[m]['dim']) for m in modalities]
        self.cache = ram_cache
    def __len__(self): return len(self.filenames)
    def __getitem__(self, idx):
        fname = self.filenames[idx]; feats = []
        for m, dim in zip(self.mods, self.dims):
            vec = self.cache.get((m, fname))
            if vec is None: vec = np.zeros(dim, dtype=np.float32)
            feats.append(vec.astype(np.float32))
        return {"features": torch.from_numpy(np.concatenate(feats)), "fname": fname}


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 7 — CMAFNet: Cross-Modal Attentive Fusion Network     ║
# ║                                                              ║
# ║  Key architectural advances over the MLP-concat baseline:   ║
# ║                                                              ║
# ║  1. Each modality becomes a TOKEN (not just a concat chunk)  ║
# ║  2. Transformer self-attention: every modality can attend   ║
# ║     to every other modality → rich cross-modal interaction  ║
# ║  3. CLS token: global context aggregated by attention       ║
# ║  4. Attention pooling: learned importance over mod tokens   ║
# ║     (vs. fixed scalar gates in the old model)               ║
# ╚══════════════════════════════════════════════════════════════╝

class ModalityEncoder(nn.Module):
    """Project one modality's pooled features to mod_proj-dim token."""
    def __init__(self, in_dim: int, mod_proj: int, dropout: float):
        super().__init__()
        mid = mod_proj * 2
        self.net = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, mid),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(mid, mod_proj),
            nn.GELU(),
            nn.LayerNorm(mod_proj),
        )
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class CrossModalBlock(nn.Module):
    """
    Pre-norm transformer block.
    Self-attention over the token sequence [CLS, mod_0, mod_1, ...].
    Every modality token can attend to every other modality — this
    is what the old concat+MLP cannot do.
    """
    def __init__(self, d: int, n_heads: int, dropout: float):
        super().__init__()
        self.ln1  = nn.LayerNorm(d)
        self.attn = nn.MultiheadAttention(d, n_heads, dropout=dropout, batch_first=True)
        self.ln2  = nn.LayerNorm(d)
        self.ff   = nn.Sequential(
            nn.Linear(d, d * 4), nn.GELU(), nn.Dropout(dropout), nn.Linear(d * 4, d)
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN self-attention
        h = self.ln1(x)
        h, _ = self.attn(h, h, h)
        x = x + self.drop(h)
        # Pre-LN feed-forward
        h = self.ln2(x)
        x = x + self.drop(self.ff(h))
        return x


class CMAFNet(nn.Module):
    """
    Cross-Modal Attentive Fusion Network (CMAFNet).

    Forward pass:
      flat  →  split per modality
            →  ModalityEncoder  (project to mod_proj dims each)
            →  [CLS | mod_0 | mod_1 | ... | mod_M]  + positional embeddings
            →  N x CrossModalBlock  (self-attention across all tokens)
            →  CLS token + attention-pooled mod tokens
            →  pre_head  →  combo_head (21-class) + sal_head (3-class)
    """
    def __init__(self, stats, modalities,
                 mod_proj: int = 64, n_heads: int = 4,
                 n_layers: int = 2, dropout: float = 0.45):
        super().__init__()
        self.mods       = modalities
        self.split_dims = [int(stats[m]['dim']) for m in modalities]
        n_mod           = len(modalities)
        d               = mod_proj
        head_dim        = d * 2   # CLS + pooled → concat

        # Per-modality encoders
        self.encoders = nn.ModuleList([
            ModalityEncoder(dim_i, d, dropout) for dim_i in self.split_dims
        ])

        # Learnable CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d))
        nn.init.trunc_normal_(self.cls_token, std=0.02)

        # Learnable per-position type embeddings
        self.pos_emb = nn.Parameter(torch.zeros(1, n_mod + 1, d))
        nn.init.trunc_normal_(self.pos_emb, std=0.02)

        # Cross-modal transformer
        self.blocks   = nn.ModuleList([CrossModalBlock(d, n_heads, dropout)
                                        for _ in range(n_layers)])
        self.final_ln = nn.LayerNorm(d)

        # Attention pooling over modality tokens (learned salience weighting)
        self.attn_pool = nn.Linear(d, 1)

        # Classification pre-head
        fuse_dim = d * 2   # [CLS | attn-pooled]
        self.pre_head = nn.Sequential(
            nn.LayerNorm(fuse_dim),
            nn.Linear(fuse_dim, head_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(head_dim),
        )

        # Output heads
        self.combo_head = nn.Linear(head_dim, N_COMBO)
        self.sal_head   = nn.Linear(head_dim, N_SAL)
        nn.init.trunc_normal_(self.combo_head.weight, std=0.02)
        nn.init.trunc_normal_(self.sal_head.weight,   std=0.02)
        nn.init.zeros_(self.combo_head.bias)
        nn.init.zeros_(self.sal_head.bias)

    def forward(self, flat: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B = flat.size(0)
        parts = torch.split(flat, self.split_dims, dim=1)

        # Encode each modality → token  [B, n_mod, d]
        mod_tokens = torch.stack(
            [self.encoders[i](p) for i, p in enumerate(parts)], dim=1
        )

        # Prepend CLS + add positional embeddings
        cls    = self.cls_token.expand(B, -1, -1)         # [B, 1, d]
        tokens = torch.cat([cls, mod_tokens], dim=1)       # [B, n_mod+1, d]
        tokens = tokens + self.pos_emb

        # Cross-modal attention: modalities interact with each other
        for block in self.blocks:
            tokens = block(tokens)
        tokens = self.final_ln(tokens)

        cls_out = tokens[:, 0]          # CLS token output  [B, d]
        mod_out = tokens[:, 1:]         # Modality tokens   [B, n_mod, d]

        # Attention pooling: learn which modalities to weight more
        attn_w  = F.softmax(self.attn_pool(mod_out), dim=1)  # [B, n_mod, 1]
        pooled  = (attn_w * mod_out).sum(1)                   # [B, d]

        fused   = torch.cat([cls_out, pooled], dim=-1)        # [B, 2d]
        h       = self.pre_head(fused)
        return self.combo_head(h), self.sal_head(h)


def count_params(m):
    n = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  Params: {n/1e6:.3f}M")
    return n

# Quick sanity check
_test_dims = {m: stats[m] for m in selected_mods}
_m = CMAFNet(_test_dims, selected_mods, mod_proj=64, n_heads=4, n_layers=2)
print("CMAFNet (cmaf_s) sanity check:")
count_params(_m)
_x = torch.zeros(4, sum(stats[m]['dim'] for m in selected_mods))
_c, _s = _m(_x)
print(f"  combo_logits: {_c.shape}  sal_logits: {_s.shape}")
del _m, _x, _c, _s


CMAFNet (cmaf_s) sanity check:
  Params: 1.253M
  combo_logits: torch.Size([4, 21])  sal_logits: torch.Size([4, 3])


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 8 — Loss functions                                    ║
# ║  NEW: Focal CE (down-weights easy examples) + Mixup support ║
# ╚══════════════════════════════════════════════════════════════╝

def focal_ce(logits, targets, gamma=2.0, weight=None, label_smoothing=0.0):
    """
    Focal cross-entropy loss.
    Multiplies CE by (1-p_t)^gamma, which:
      - Down-weights easy/well-classified examples (p_t -> 1)
      - Focuses training on hard/rare examples (p_t -> 0)
    gamma=2 is standard; set to 0 to recover vanilla CE.
    """
    ce = F.cross_entropy(logits, targets, weight=weight,
                         label_smoothing=label_smoothing, reduction='none')
    pt = torch.exp(-ce.detach())          # probability of correct class
    return ((1.0 - pt) ** gamma * ce).mean()


def soft_cross_entropy(logits, soft_targets):
    """Cross-entropy with soft (mixup) targets."""
    log_p = F.log_softmax(logits, dim=-1)
    return -(soft_targets * log_p).sum(-1).mean()


def make_soft_labels(y, n_classes, smoothing=0.0):
    """One-hot + optional label smoothing, suitable for mixup."""
    B = y.size(0)
    s = torch.full((B, n_classes), smoothing / max(n_classes - 1, 1), device=y.device)
    s.scatter_(1, y.unsqueeze(1), 1.0 - smoothing)
    return s


def compute_combo_weights(df) -> torch.Tensor:
    """Inverse-frequency class weights for the 21 combo classes."""
    counts = np.zeros(N_COMBO, dtype=np.float32)
    for idx in df['combo_idx'].values:
        counts[int(idx)] += 1
    counts = np.maximum(counts, 1)
    w = (len(df) / (N_COMBO * counts)).clip(0.5, 10.)
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)


def compute_loss(combo_logits, sal_logits,
                 combo_targets, sal_targets, sal_mask,
                 combo_soft=None, sal_soft=None,
                 combo_weights=None, gamma=2.0, label_smoothing=0.05):
    """
    Multi-task loss:
      - Focal CE for 21-class combo prediction
      - Focal CE for 3-class salience (blended samples only)
    Supports soft (mixup) targets for both heads.
    Salience weight 0.7: presence is the foundation for salience.
    """
    if combo_soft is not None:
        combo_loss = soft_cross_entropy(combo_logits, combo_soft)
    else:
        combo_loss = focal_ce(combo_logits, combo_targets,
                               gamma=gamma, weight=combo_weights,
                               label_smoothing=label_smoothing)

    if sal_mask.sum() > 0:
        blend_idx = sal_mask > 0.5
        if sal_soft is not None:
            sal_loss = soft_cross_entropy(sal_logits[blend_idx], sal_soft[blend_idx])
        else:
            sal_loss = focal_ce(sal_logits[blend_idx], sal_targets[blend_idx],
                                 gamma=1.5, label_smoothing=label_smoothing)
    else:
        sal_loss = combo_logits.new_tensor(0.)

    total = combo_loss + 0.7 * sal_loss
    return total, combo_loss, sal_loss


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 9 — Decoding & official metrics (unchanged)           ║
# ╚══════════════════════════════════════════════════════════════╝
def decode_sample(combo_idx: int, sal_idx: int) -> List[Dict]:
    combo = ALL_COMBOS[combo_idx]
    if len(combo) == 1:
        return [{'emotion': combo[0], 'salience': 100.}]
    ea, eb = combo[0], combo[1]
    if   sal_idx == 0: sa, sb = 70., 30.
    elif sal_idx == 1: sa, sb = 50., 50.
    else:              sa, sb = 30., 70.
    return [{'emotion': ea, 'salience': sa}, {'emotion': eb, 'salience': sb}]

def decode_batch(combo_probs, sal_probs, fnames):
    pred = {}
    for i, fn in enumerate(fnames):
        key       = fn if fn.endswith('.mov') else fn + '.mov'
        pred[key] = decode_sample(int(np.argmax(combo_probs[i])),
                                   int(np.argmax(sal_probs[i])))
    return pred

def build_truth(df):
    truth = {}
    for _, row in df.iterrows():
        fn  = str(row['filename'])
        key = fn if fn.endswith('.mov') else fn + '.mov'
        truth[key] = decode_sample(int(row['combo_idx']), int(row['sal_idx']))
    return truth

def official_scores(pred_map: Dict, df: pd.DataFrame) -> Tuple[float, float, float]:
    """
    ACC_presence: emotion SET exactly matches (all samples).
    ACC_salience: SET + salience split match (blended samples only,
                  presence must also be correct).
    """
    truth  = build_truth(df)
    acc_p, acc_s = [], []
    for key, gt in truth.items():
        pr   = pred_map.get(key, [])
        gt_e = sorted(x['emotion'] for x in gt)
        pr_e = sorted(x['emotion'] for x in pr)
        p_ok = int(gt_e == pr_e)
        acc_p.append(p_ok)
        if len(gt) == 2:
            if p_ok:
                gt_d = {x['emotion']: round(float(x['salience'])) for x in gt}
                pr_d = {x['emotion']: round(float(x['salience'])) for x in pr}
                acc_s.append(int(gt_d == pr_d))
            else:
                acc_s.append(0)
    a = float(np.mean(acc_p)) if acc_p else 0.
    b = float(np.mean(acc_s)) if acc_s else 0.
    return a, b, 0.5 * (a + b)


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 10 — Training utilities                               ║
# ║  NEW: Feature-space Mixup augmentation                      ║
# ╚══════════════════════════════════════════════════════════════╝
def make_sample_weights(df):
    mix = df['sal_mask'].values.astype(np.float64)
    return torch.as_tensor(np.where(mix > 0.5, MIX_SAMPLE_MULT, 1.), dtype=torch.double)

def get_amp():
    if DEVICE.type == 'cuda':
        try: return torch.amp.autocast('cuda')
        except: return torch.cuda.amp.autocast()
    return nullcontext()

def get_scaler():
    if DEVICE.type == 'cuda':
        try: return torch.amp.GradScaler('cuda')
        except: return torch.cuda.amp.GradScaler(enabled=True)
    return None

def build_cosine_schedule(optimizer, warmup_steps, total_steps, min_lr=0.05):
    def fn(step):
        if step < warmup_steps: return (step + 1) / warmup_steps
        p = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return max(min_lr, 0.5 * (1. + math.cos(math.pi * p)))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)

def make_loader(ds, shuffle=False, sampler=None):
    """Always creates a FRESH DataLoader — avoids iterator-exhaustion bugs."""
    return DataLoader(ds, BATCH_SIZE, shuffle=shuffle, sampler=sampler,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == 'cuda'))


def apply_mixup(feats, combo_idx, sal_idx, sal_mask):
    """
    Feature-space Mixup (applied to ~50% of batches).

    Samples lambda from Beta(alpha, alpha) and linearly interpolates
    feature vectors and their soft labels.  This creates virtual
    training points between real samples — effective regularization
    for small datasets.

    Returns (mixed_feats, combo_soft, sal_soft, mixed_sal_mask)
    or (feats, None, None, sal_mask) if Mixup is skipped.
    """
    if MIXUP_ALPHA <= 0 or random.random() > 0.5:
        return feats, None, None, sal_mask

    B    = feats.size(0)
    lam  = torch.from_numpy(
        np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA, size=B).astype(np.float32)
    ).to(feats.device)
    perm = torch.randperm(B, device=feats.device)

    mixed = lam[:, None] * feats + (1 - lam[:, None]) * feats[perm]

    # Soft combo labels
    s1c = make_soft_labels(combo_idx,       N_COMBO, smoothing=0.05)
    s2c = make_soft_labels(combo_idx[perm], N_COMBO, smoothing=0.05)
    combo_soft = lam[:, None] * s1c + (1 - lam[:, None]) * s2c

    # Soft salience labels
    s1s = make_soft_labels(sal_idx,       N_SAL, smoothing=0.05)
    s2s = make_soft_labels(sal_idx[perm], N_SAL, smoothing=0.05)
    sal_soft = lam[:, None] * s1s + (1 - lam[:, None]) * s2s

    # Mixed sal mask: blended if EITHER sample is blended
    mixed_sal_mask = torch.max(sal_mask, sal_mask[perm])
    return mixed, combo_soft, sal_soft, mixed_sal_mask


def train_epoch(model, ds_tr, opt, sched, scaler, combo_w):
    model.train()
    total, n = 0., 0
    sampler  = WeightedRandomSampler(make_sample_weights(ds_tr.df), len(ds_tr), replacement=True)
    loader   = make_loader(ds_tr, sampler=sampler)
    opt.zero_grad(set_to_none=True)

    for batch in loader:
        feats  = batch['features'].to(DEVICE, non_blocking=True)
        c_idx  = batch['combo_idx'].to(DEVICE, non_blocking=True)
        s_idx  = batch['sal_idx'].to(DEVICE,   non_blocking=True)
        s_mask = batch['sal_mask'].to(DEVICE,  non_blocking=True)

        mixed, combo_soft, sal_soft, eff_mask = apply_mixup(feats, c_idx, s_idx, s_mask)

        with get_amp():
            cl, sl   = model(mixed)
            loss, *_ = compute_loss(cl, sl, c_idx, s_idx, eff_mask,
                                     combo_soft=combo_soft, sal_soft=sal_soft,
                                     combo_weights=combo_w)
        if scaler:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(opt); scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if sched: sched.step()
        opt.zero_grad(set_to_none=True)
        total += loss.item() * feats.size(0); n += feats.size(0)
    return total / max(n, 1)


@torch.no_grad()
def val_epoch(model, ds_va):
    """Creates a FRESH DataLoader — no iterator-exhaustion issue."""
    model.eval()
    loader = make_loader(ds_va, shuffle=False)
    cp, sp, fns = [], [], []
    for batch in loader:
        feats = batch['features'].to(DEVICE, non_blocking=True)
        with get_amp():
            cl, sl = model(feats)
        cp.append(torch.softmax(cl, dim=1).cpu().numpy())
        sp.append(torch.softmax(sl, dim=1).cpu().numpy())
        fns.extend(batch['fname'])
    return np.vstack(cp), np.vstack(sp), fns


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 11 — run_fold with SWA                                ║
# ╚══════════════════════════════════════════════════════════════╝
def run_fold(fold, cfg, meta_df, stats, ram_cache, epochs=EPOCHS):
    name = cfg['name']
    print(f"\n{'─'*60}\nFold {fold} | {name}\n{'─'*60}")

    ds_tr = BlEmoDataset(meta_df, selected_mods, fold, 'train', stats,
                          ram_cache, modality_drop=0.15, feat_noise=0.01)
    ds_va = BlEmoDataset(meta_df, selected_mods, fold, 'val',   stats,
                          ram_cache, modality_drop=0.,   feat_noise=0.)

    model = CMAFNet(
        stats, selected_mods,
        mod_proj=cfg['mod_proj'], n_heads=cfg['n_heads'],
        n_layers=cfg['n_layers'], dropout=cfg['dropout'],
    ).to(DEVICE)
    count_params(model)
    swa_model = AveragedModel(model)

    steps_per_ep = max(1, math.ceil(len(ds_tr) / BATCH_SIZE))
    total_steps  = epochs * steps_per_ep
    warmup_steps = WARMUP_EPOCHS * steps_per_ep

    opt     = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=WEIGHT_DECAY)
    sched   = build_cosine_schedule(opt, warmup_steps, total_steps)
    swa_s   = SWALR(opt, swa_lr=SWA_LR, anneal_epochs=5)
    scaler  = get_scaler()
    combo_w = compute_combo_weights(ds_tr.df)

    best_score, best_state = -1., None
    hist, start_ep = [], 1

    best_ckpt = f"{OUTPUT_DIR}/models/{name}_fold{fold}.pt"
    if os.path.exists(best_ckpt):
        ck = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
        try:
            model.load_state_dict(ck['model_state'], strict=False)
            best_score = ck.get('best_score', -1.)
            hist       = ck.get('history', [])
            start_ep   = len(hist) + 1
            print(f"  Resuming from ep{start_ep-1} (best={best_score:.4f})")
        except Exception as e:
            print(f"  Could not resume ({e}) — training from scratch")
        if start_ep > epochs:
            print(f"  Already complete. Returning existing checkpoint.")
            return best_ckpt, best_score

    for ep in range(start_ep, epochs + 1):
        t0        = time.time()
        using_swa = ep >= SWA_START

        if ep == SWA_START:
            for g in opt.param_groups: g['lr'] = SWA_LR
            print(f"  [ep{ep}] ─── SWA started (lr={SWA_LR}) ───")

        tr_loss = train_epoch(model, ds_tr, opt,
                               None if using_swa else sched, scaler, combo_w)
        if using_swa:
            swa_model.update_parameters(model)
            swa_s.step()

        cp, sp, fns = val_epoch(model, ds_va)
        pm          = decode_batch(cp, sp, fns)
        ap, asal, score = official_scores(pm, ds_va.df)

        tag = "[SWA] " if using_swa else "      "
        print(f"  ep{ep:3d}/{epochs} {tag}loss={tr_loss:.4f}  "
              f"pres={ap:.4f}  sal={asal:.4f}  avg={score:.4f}  [{time.time()-t0:.0f}s]")

        if score > best_score:
            best_score = score
            best_state = copy.deepcopy(model.state_dict())
            print("    ↑ new best!")

        hist.append(dict(ep=ep, loss=tr_loss, pres=ap, sal=asal, avg=score))

    # ── SWA final evaluation ──────────────────────────────────────────
    if SWA_START <= epochs:
        print("  Updating SWA batch-norm stats ...")
        update_bn(make_loader(ds_tr, shuffle=True), swa_model, device=DEVICE)
        cp_s, sp_s, fns_s = val_epoch(swa_model.module, ds_va)
        pm_s              = decode_batch(cp_s, sp_s, fns_s)
        ap_s, asal_s, sc_s = official_scores(pm_s, ds_va.df)
        print(f"  SWA: pres={ap_s:.4f}  sal={asal_s:.4f}  avg={sc_s:.4f}")
        if sc_s > best_score:
            best_score = sc_s
            best_state = copy.deepcopy(swa_model.module.state_dict())
            print("    ↑ SWA is best!")

    torch.save(dict(
        model_state=best_state, stats=stats,
        modalities=selected_mods, cfg=cfg,
        best_score=best_score, history=hist
    ), best_ckpt)
    print(f"  ✓ Saved → {best_ckpt}  (score={best_score:.4f})")

    del model, swa_model
    if DEVICE.type == 'cuda': torch.cuda.empty_cache()
    gc.collect()
    return best_ckpt, best_score


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 12 — Train all variants × all folds                  ║
# ╚══════════════════════════════════════════════════════════════╝
best_ckpts = []
for cfg in MODEL_VARIANTS:
    for f in FOLDS_TO_TRAIN:
        path, score = run_fold(f, cfg, meta_df, stats, train_cache)
        if path:
            best_ckpts.append(dict(model=cfg['name'], fold=f, ckpt=path, score=score))

best_ckpts_sorted = sorted(best_ckpts, key=lambda x: x['score'], reverse=True)
print("\nAll checkpoints (sorted):")
for b in best_ckpts_sorted:
    print(f"  {b['model']:10s}  fold={b['fold']}  score={b['score']:.4f}")



────────────────────────────────────────────────────────────
Fold 0 | cmaf_s
────────────────────────────────────────────────────────────
  [train fold=0] n=1934  feat_len=8448
  [val fold=0] n=522  feat_len=8448
  Params: 1.253M
  ep  1/80       loss=3.4978  pres=0.0766  sal=0.0393  avg=0.0580  [3s]
    ↑ new best!
  ep  2/80       loss=3.5090  pres=0.1054  sal=0.0607  avg=0.0830  [2s]
    ↑ new best!
  ep  3/80       loss=3.4237  pres=0.1092  sal=0.0714  avg=0.0903  [2s]
    ↑ new best!
  ep  4/80       loss=3.2157  pres=0.1054  sal=0.0607  avg=0.0830  [2s]
  ep  5/80       loss=3.1426  pres=0.1092  sal=0.0643  avg=0.0867  [2s]
  ep  6/80       loss=3.0428  pres=0.1609  sal=0.0929  avg=0.1269  [2s]
    ↑ new best!
  ep  7/80       loss=3.1231  pres=0.1609  sal=0.1071  avg=0.1340  [2s]
    ↑ new best!
  ep  8/80       loss=2.7787  pres=0.1877  sal=0.1071  avg=0.1474  [1s]
    ↑ new best!
  ep  9/80       loss=2.7458  pres=0.2241  sal=0.1214  avg=0.1728  [2s]
    ↑ new best!
  ep 10/8

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 13 — Inference & ensemble helpers                     ║
# ╚══════════════════════════════════════════════════════════════╝
def load_model_ckpt(path):
    ck  = torch.load(path, map_location=DEVICE, weights_only=False)
    cfg = ck['cfg']
    m   = CMAFNet(ck['stats'], ck['modalities'],
                   mod_proj=cfg['mod_proj'], n_heads=cfg['n_heads'],
                   n_layers=cfg['n_layers'], dropout=cfg['dropout'])
    m.load_state_dict(ck['model_state'])
    m.to(DEVICE).eval()
    return m


@torch.no_grad()
def predict_one_model(model, loader):
    cp, sp, fns = [], [], []
    for batch in loader:
        feats = batch['features'].to(DEVICE, non_blocking=True)
        with get_amp():
            cl, sl = model(feats)
        cp.append(torch.softmax(cl, dim=1).cpu().numpy())
        sp.append(torch.softmax(sl, dim=1).cpu().numpy())
        fns.extend(batch['fname'])
    return np.vstack(cp), np.vstack(sp), fns


def ensemble_predict(items, ds):
    """
    Score²-weighted average ensemble.
    Creates a FRESH DataLoader per model — avoids iterator exhaustion.
    """
    w = np.array([x['score'] ** 2 for x in items], dtype=np.float64)
    w /= w.sum()
    cs = ss = fns_ref = None
    for item, wi in zip(items, w):
        model  = load_model_ckpt(item['ckpt'])
        loader = make_loader(ds, shuffle=False)   # FRESH loader
        cp, sp, fns = predict_one_model(model, loader)
        del model
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        if fns_ref is None: fns_ref = fns; cs = cp * wi; ss = sp * wi
        else:                               cs += cp * wi; ss += sp * wi
    return cs, ss, fns_ref


In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 14 — True out-of-fold CV validation                   ║
# ║                                                              ║
# ║  For fold v: ONLY use models with item['fold'] == v         ║
# ║  These trained on folds != v, so fold v's data is truly OOF ║
# ╚══════════════════════════════════════════════════════════════╝
def cv_validation(all_ckpts, meta_df, stats, ram_cache):
    fold_results = []
    for fold in sorted(meta_df['fold'].unique()):
        oof = sorted([x for x in all_ckpts if x['fold'] == fold],
                     key=lambda x: x['score'], reverse=True)
        if not oof:
            print(f"  Fold {fold}: no checkpoints — skipping")
            continue
        ds_va = BlEmoDataset(meta_df, selected_mods, fold, 'val',
                              stats, ram_cache, 0., 0.)
        cs, ss, fns = ensemble_predict(oof, ds_va)
        pm          = decode_batch(cs, ss, fns)
        ap, asal, avg = official_scores(pm, ds_va.df)
        names = [x['model'] for x in oof]
        print(f"  Fold {fold} [{len(oof)} models: {', '.join(names)}]")
        print(f"    pres={ap:.4f}  sal={asal:.4f}  avg={avg:.4f}")
        fold_results.append((ap, asal, avg))

    if not fold_results:
        return 0., 0., 0.

    mean_p   = float(np.mean([r[0] for r in fold_results]))
    mean_sal = float(np.mean([r[1] for r in fold_results]))
    mean_avg = float(np.mean([r[2] for r in fold_results]))

    print(f"\n{'═'*55}")
    print(f"CV MEAN (true OOF)   pres={mean_p:.4f}  sal={mean_sal:.4f}  avg={mean_avg:.4f}")
    print(f"\nBaseline reference:")
    print(f"  ImageBind+WavLM    val pres=0.345  sal=0.170  avg=0.258")
    print(f"  VideoMAEv2+HuBERT  val pres=0.332  sal=0.138  avg=0.235")
    print(f"  HiCMAE             val pres=0.298  sal=0.180  avg=0.239")
    print(f"  Trivial baseline   val pres=0.077  sal=0.000  avg=0.039")
    if mean_p   > 0.345: print(f"\n  ✅ BEATS presence baseline (ImageBind+WavLM 0.345)!")
    if mean_sal > 0.180: print(f"  ✅ BEATS salience baseline (HiCMAE 0.180)!")
    return mean_p, mean_sal, mean_avg


cv_validation(best_ckpts, meta_df, stats, train_cache)


  [val fold=0] n=522  feat_len=8448
  Fold 0 [4 models: cmaf_xl, cmaf_m, cmaf_s, cmaf_l]
    pres=0.3927  sal=0.1429  avg=0.2678
  [val fold=1] n=486  feat_len=8448
  Fold 1 [4 models: cmaf_xl, cmaf_s, cmaf_m, cmaf_l]
    pres=0.3992  sal=0.1556  avg=0.2774
  [val fold=2] n=468  feat_len=8448
  Fold 2 [4 models: cmaf_xl, cmaf_s, cmaf_l, cmaf_m]
    pres=0.4081  sal=0.1250  avg=0.2666
  [val fold=3] n=472  feat_len=8448
  Fold 3 [4 models: cmaf_m, cmaf_s, cmaf_l, cmaf_xl]
    pres=0.3877  sal=0.1458  avg=0.2668
  [val fold=4] n=508  feat_len=8448
  Fold 4 [4 models: cmaf_m, cmaf_xl, cmaf_s, cmaf_l]
    pres=0.3150  sal=0.0845  avg=0.1997

═══════════════════════════════════════════════════════
CV MEAN (true OOF)   pres=0.3805  sal=0.1307  avg=0.2556

Baseline reference:
  ImageBind+WavLM    val pres=0.345  sal=0.170  avg=0.258
  VideoMAEv2+HuBERT  val pres=0.332  sal=0.138  avg=0.235
  HiCMAE             val pres=0.298  sal=0.180  avg=0.239
  Trivial baseline   val pres=0.077  sal=0.000

(0.38053788273872363, 0.13074109824109822, 0.2556394904899109)

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  Cell 15 — Test submission (ensemble of top-K models)       ║
# ║  Test data was never seen by any model — use all freely.    ║
# ╚══════════════════════════════════════════════════════════════╝
test_ensemble = best_ckpts_sorted[:ENSEMBLE_TOPK]
print(f"Test ensemble: {len(test_ensemble)} models")
for b in test_ensemble:
    print(f"  {b['model']:10s}  fold={b['fold']}  score={b['score']:.4f}")

test_ds     = BlEmoTestDataset(test_samples, selected_mods, stats, test_cache)
cs, ss, fns = ensemble_predict(test_ensemble, test_ds)
pred_map    = decode_batch(cs, ss, fns)

out = f"{OUTPUT_DIR}/submission.json"
with open(out, 'w') as f:
    json.dump({'predictions': pred_map}, f, indent=2)
zp = out.replace('.json', '.zip')
with zipfile.ZipFile(zp, 'w') as z:
    z.write(out, os.path.basename(out))

print(f"\nSubmission files:")
print(f"  JSON: {out}")
print(f"  ZIP:  {zp}")
print(f"  Total predictions: {len(pred_map)}")
sample_key = list(pred_map.keys())[0]
print(f"\nSample: {sample_key} -> {pred_map[sample_key]}")


Test ensemble: 12 models
  cmaf_xl     fold=0  score=0.2803
  cmaf_xl     fold=2  score=0.2739
  cmaf_m      fold=0  score=0.2723
  cmaf_s      fold=2  score=0.2706
  cmaf_xl     fold=1  score=0.2693
  cmaf_l      fold=2  score=0.2644
  cmaf_m      fold=2  score=0.2614
  cmaf_m      fold=3  score=0.2613
  cmaf_s      fold=0  score=0.2575
  cmaf_s      fold=1  score=0.2572
  cmaf_m      fold=1  score=0.2560
  cmaf_s      fold=3  score=0.2541

Submission files:
  JSON: /content/drive/MyDrive/blemore_project-backup/blemore_output-cmaf-01/submission.json
  ZIP:  /content/drive/MyDrive/blemore_project-backup/blemore_output-cmaf-01/submission.zip
  Total predictions: 594

Sample: test_0001.mov -> [{'emotion': 'ang', 'salience': 30.0}, {'emotion': 'disg', 'salience': 70.0}]
